# Topic 10: Graphs (BFS, DFS, Shortest Path)

**Goal**: Understand graph representations and master traversal + shortest path algorithms.  
**Time**: ~8-10 hours  
**Prereqs**: Topics 4-6, 9

---

## Why Graphs?

Graphs model **relationships between things**:
- **Social networks** — friends of friends
- **Maps** — shortest route between cities
- **Dependencies** — build systems, course prerequisites
- Many problems that don't *look* like graphs **are** graphs (word ladder, number of islands, mazes)

```
    A --- B
    |   / |
    |  /  |
    C --- D --- E

Adjacency List:
  A: [B, C]
  B: [A, C, D]
  C: [A, B, D]
  D: [B, C, E]
  E: [D]
```

A graph is just **nodes** (vertices) and **edges** (connections).  
Everything else — BFS, DFS, Dijkstra, topological sort — is a technique for *walking* that structure.

---

## Part 1: Graph Basics

### Three Axes of Classification

```
DIRECTED vs UNDIRECTED:

  Undirected          Directed (digraph)
  A --- B             A --→ B
  |     |             ↑     |
  C --- D             C ←-- D

  edge(A,B) = edge(B,A)    edge(A,B) ≠ edge(B,A)
  "friendship"              "follows on Twitter"

-----------------------------------------------

WEIGHTED vs UNWEIGHTED:

  Unweighted          Weighted
  A --- B             A -5- B
  |     |             |     |
  C --- D             3     2
                      |     |
                      C -1- D

  all edges equal      edges have costs/distances
  "connected?"         "how far?"

-----------------------------------------------

CYCLIC vs ACYCLIC:

  Cyclic              Acyclic (DAG if directed)
  A → B               A → B
  ↑   ↓               ↓   ↓
  D ← C               C   D

  can revisit nodes    no path leads back to itself
  "deadlocks"          "dependency chains"
```

In [ ]:
from collections import defaultdict

edges = [("A","B"), ("A","C"), ("B","C"), ("B","D"), ("C","D"), ("D","E")]

graph = defaultdict(list)
for u, v in edges:
    graph[u].append(v)
    graph[v].append(u)

print("=== Adjacency List (undirected) ===")
for node in sorted(graph):
    print(f"  {node}: {sorted(graph[node])}")

print()

directed_edges = [("A","B"), ("A","C"), ("B","D"), ("D","C")]
digraph = defaultdict(list)
for u, v in directed_edges:
    digraph[u].append(v)

print("=== Adjacency List (directed) ===")
for node in ["A", "B", "C", "D"]:
    print(f"  {node} → {digraph[node]}")

In [ ]:
nodes = ["A", "B", "C", "D", "E"]
n = len(nodes)
idx = {ch: i for i, ch in enumerate(nodes)}

matrix = [[0] * n for _ in range(n)]
for u, v in edges:
    matrix[idx[u]][idx[v]] = 1
    matrix[idx[v]][idx[u]] = 1

print("=== Adjacency Matrix ===")
print("    " + "  ".join(nodes))
for i, node in enumerate(nodes):
    print(f"  {node} {matrix[i]}")

print()
print(f"  Edge A-B? matrix[A][B] = {matrix[idx['A']][idx['B']]}")
print(f"  Edge A-E? matrix[A][E] = {matrix[idx['A']][idx['E']]}")

---

### Representation Comparison

| | Adjacency List | Adjacency Matrix |
|---|---|---|
| Space | O(V + E) | O(V²) |
| Check edge exists? | O(degree) | O(1) |
| Get all neighbors | O(degree) | O(V) |
| Add edge | O(1) | O(1) |
| Best for | Sparse graphs (most problems) | Dense graphs, quick edge lookup |

**Rule of thumb**: Use adjacency list (`dict` of `list`) for 95% of interview problems.

---

## Part 2: BFS (Breadth-First Search)

**Explore level by level, like ripples in a pond.**  
Uses a **QUEUE** — process the oldest discovered node first.

```
    A --- B
    |   / |
    |  /  |
    C --- D --- E

BFS from A:

  Level 0: [A]              ← start
           queue: [A]
           visited: {A}

  Level 1: [B, C]           ← neighbors of A
           queue: [B, C]
           visited: {A, B, C}

  Level 2: [D]              ← neighbors of B,C not yet visited
           queue: [D]
           visited: {A, B, C, D}

  Level 3: [E]              ← neighbor of D
           queue: [E]
           visited: {A, B, C, D, E}
```

Key insight: BFS guarantees **shortest path** in an unweighted graph  
(the first time you reach a node is always via the fewest edges).

In [ ]:
from collections import deque

def bfs(graph, start):
    visited = {start}
    queue = deque([start])
    level = 0
    order = []

    while queue:
        level_size = len(queue)
        level_nodes = []

        for _ in range(level_size):
            node = queue.popleft()
            level_nodes.append(node)
            for neighbor in sorted(graph[node]):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)

        print(f"  Level {level}: {level_nodes}")
        order.extend(level_nodes)
        level += 1

    return order

graph = defaultdict(list)
for u, v in [("A","B"),("A","C"),("B","C"),("B","D"),("C","D"),("D","E")]:
    graph[u].append(v)
    graph[v].append(u)

print("=== BFS from A ===")
result = bfs(graph, "A")
print(f"  Visit order: {result}")

In [ ]:
def bfs_shortest_path(graph, start, end):
    """Shortest path in unweighted graph via BFS."""
    if start == end:
        return [start]

    visited = {start}
    queue = deque([(start, [start])])

    while queue:
        node, path = queue.popleft()
        for neighbor in sorted(graph[node]):
            if neighbor not in visited:
                new_path = path + [neighbor]
                if neighbor == end:
                    return new_path
                visited.add(neighbor)
                queue.append((neighbor, new_path))

    return None

print("=== Shortest Paths (unweighted) ===")
for target in ["B", "C", "D", "E"]:
    path = bfs_shortest_path(graph, "A", target)
    print(f"  A → {target}: {' → '.join(path)}  (length {len(path)-1})")

---

## Part 3: DFS (Depth-First Search)

**Go as deep as possible, then backtrack.**  
Uses a **STACK** (explicit or via recursion).

```
    A --- B
    |   / |
    |  /  |
    C --- D --- E

DFS from A (recursive, alphabetical order):

  visit A
  ├─ visit B               (A's neighbor)
  │  ├─ visit C            (B's unvisited neighbor)
  │  │  └─ visit D         (C's unvisited neighbor)
  │  │     └─ visit E      (D's unvisited neighbor)
  │  │        └─ backtrack (E has no unvisited neighbors)
  │  └─ backtrack
  └─ backtrack

  Order: A → B → C → D → E
```

In [ ]:
def dfs_iterative(graph, start):
    visited = set()
    stack = [start]
    order = []

    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        order.append(node)
        print(f"  visit {node}  |  stack after: {stack}")

        for neighbor in sorted(graph[node], reverse=True):
            if neighbor not in visited:
                stack.append(neighbor)

    return order

print("=== DFS Iterative from A ===")
result = dfs_iterative(graph, "A")
print(f"  Order: {result}")

In [ ]:
def dfs_recursive(graph, node, visited=None, depth=0):
    if visited is None:
        visited = set()

    visited.add(node)
    indent = "  │ " * depth
    print(f"  {indent}visit {node}")

    for neighbor in sorted(graph[node]):
        if neighbor not in visited:
            dfs_recursive(graph, neighbor, visited, depth + 1)

    if depth > 0:
        back_indent = "  │ " * (depth - 1)
        print(f"  {back_indent}└─ backtrack from {node}")

    return visited

print("=== DFS Recursive from A ===")
dfs_recursive(graph, "A")

---

### BFS vs DFS — When to Use Which

| | BFS | DFS |
|---|---|---|
| Data structure | Queue | Stack / Recursion |
| Explores | Level by level | Deep first |
| Shortest path? | **Yes** (unweighted) | No |
| Memory | O(width of graph) | O(depth of graph) |
| Use when | Shortest path, level-order | Cycle detection, connected components |

```
BFS explores like ripples:        DFS explores like a maze:

       1                                  1
      / \                                / 
     2   3                              2   
    / \   \                            /   
   4   5   6                          3     
                                     /       
                                    4         
  order: 1,2,3,4,5,6              order: 1,2,3,4...backtrack...5,6
```

---

## Problem 1: Number of Islands — LC #200

Given a 2D grid of `'1'` (land) and `'0'` (water), count the number of islands.  
An island is surrounded by water and formed by connecting adjacent lands (up/down/left/right).

```
Grid:               Graph view:
  1 1 0 0 0          (0,0)-(0,1)     
  1 1 0 0 0          |     |          
  0 0 1 0 0         (1,0)-(1,1)   (2,2)     (3,3)-(3,4)
  0 0 0 1 1                                   |     |
                                              (4,3)  ...
  Island 1: top-left block of 1s
  Island 2: center 1
  Island 3: bottom-right block of 1s
  Answer: 3
```

**Strategy**: Scan every cell. When you find an unvisited `'1'`, that's a new island.  
DFS/BFS from it to mark all connected land as visited.

In [ ]:
def numIslands(grid):
    if not grid:
        return 0

    rows, cols = len(grid), len(grid[0])
    count = 0

    def dfs(r, c, island_id):
        if r < 0 or r >= rows or c < 0 or c >= cols:
            return
        if grid[r][c] != "1":
            return
        grid[r][c] = "#"
        print(f"    marking ({r},{c}) as island {island_id}")
        for dr, dc in [(0,1),(0,-1),(1,0),(-1,0)]:
            dfs(r+dr, c+dc, island_id)

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1":
                count += 1
                print(f"  Found island {count} starting at ({r},{c})")
                dfs(r, c, count)

    return count

grid = [
    ["1","1","0","0","0"],
    ["1","1","0","0","0"],
    ["0","0","1","0","0"],
    ["0","0","0","1","1"],
]

print("=== Number of Islands ===")
result = numIslands(grid)
print(f"\n  Total islands: {result}")

---

## Problem 2: Clone Graph — LC #133

Given a reference to a node in a connected undirected graph, return a **deep copy**.  
Each node has a `val` and a list of `neighbors`.

```
Original:         Clone:
  1 --- 2           1' --- 2'
  |     |           |      |
  4 --- 3           4' --- 3'

Strategy: BFS/DFS with a hash map
  old_node → new_node (clone)

  Visit 1 → create 1'
  Visit 2 → create 2', link 1'↔2'
  Visit 3 → create 3', link 2'↔3'
  Visit 4 → create 4', link 1'↔4', 3'↔4'
```

In [ ]:
class GraphNode:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors else []

def cloneGraph(node):
    if not node:
        return None

    cloned = {node: GraphNode(node.val)}
    queue = deque([node])
    print(f"  clone node {node.val}")

    while queue:
        curr = queue.popleft()
        for neighbor in curr.neighbors:
            if neighbor not in cloned:
                cloned[neighbor] = GraphNode(neighbor.val)
                print(f"  clone node {neighbor.val}")
                queue.append(neighbor)
            cloned[curr].neighbors.append(cloned[neighbor])
            print(f"    link {curr.val}' → {neighbor.val}'")

    return cloned[node]

n1, n2, n3, n4 = GraphNode(1), GraphNode(2), GraphNode(3), GraphNode(4)
n1.neighbors = [n2, n4]
n2.neighbors = [n1, n3]
n3.neighbors = [n2, n4]
n4.neighbors = [n1, n3]

print("=== Clone Graph ===")
clone = cloneGraph(n1)
print(f"\n  Original node 1 id: {id(n1)}")
print(f"  Cloned node 1 id:   {id(clone)}")
print(f"  Same object? {n1 is clone}")
print(f"  Clone neighbors: {[n.val for n in clone.neighbors]}")

---

## Problem 3: Course Schedule (Cycle Detection) — LC #207

There are `numCourses` courses labeled `0` to `n-1`.  
`prerequisites[i] = [a, b]` means: to take course `a`, you must first take `b`.  
Return `True` if you can finish all courses (i.e., no cycles).

```
prerequisites = [[1,0], [2,1], [3,2], [1,3]]

Graph:  0 → 1 → 2 → 3
             ↑         |
             └─────────┘    ← CYCLE!

Answer: False (can't finish)
```

### 3-Color DFS for Cycle Detection

```
WHITE (0) = unvisited
GRAY  (1) = in current DFS path (being explored)
BLACK (2) = fully processed

If we visit a GRAY node → CYCLE found!
  (we've reached a node that's still in our current path)

DFS trace on above graph:
  visit 0 (WHITE→GRAY)
  ├─ visit 1 (WHITE→GRAY)
  │  ├─ visit 2 (WHITE→GRAY)
  │  │  ├─ visit 3 (WHITE→GRAY)
  │  │  │  └─ neighbor 1 is GRAY → CYCLE!
```

In [ ]:
def canFinish(numCourses, prerequisites):
    graph = defaultdict(list)
    for course, prereq in prerequisites:
        graph[prereq].append(course)

    WHITE, GRAY, BLACK = 0, 1, 2
    color = [WHITE] * numCourses

    def dfs(node, depth=0):
        color[node] = GRAY
        indent = "  │ " * depth
        print(f"  {indent}visit {node} (→ GRAY)")

        for neighbor in graph[node]:
            if color[neighbor] == GRAY:
                n_indent = "  │ " * (depth + 1)
                print(f"  {n_indent}neighbor {neighbor} is GRAY → CYCLE!")
                return False
            if color[neighbor] == WHITE:
                if not dfs(neighbor, depth + 1):
                    return False

        color[node] = BLACK
        print(f"  {indent}done  {node} (→ BLACK)")
        return True

    for course in range(numCourses):
        if color[course] == WHITE:
            if not dfs(course):
                return False
    return True

print("=== Course Schedule — Has Cycle ===")
print(f"  Result: {canFinish(4, [[1,0],[2,1],[3,2],[1,3]])}")

print()
print("=== Course Schedule — No Cycle ===")
print(f"  Result: {canFinish(4, [[1,0],[2,0],[3,1],[3,2]])}")

---

## Problem 4: Topological Sort — LC #210

Order nodes so that every directed edge `u → v` has `u` before `v`.  
Only works on **DAGs** (Directed Acyclic Graphs).

```
Courses:  0 ← 1 ← 2
          0 ← 3

Graph (prerequisite → course):
  2 → 1 → 0
  3 → 0

Topological orders:
  [2, 1, 3, 0]   ✓  (both 1 and 3 before 0, 2 before 1)
  [3, 2, 1, 0]   ✓
  [2, 3, 1, 0]   ✓
  [1, 2, 3, 0]   ✗  (2 must come before 1)
```

### Two Approaches

```
KAHN'S ALGORITHM (BFS + indegree):
  1. Count incoming edges (indegree) for each node
  2. Add all nodes with indegree 0 to queue
  3. Process queue: remove node, reduce indegree of neighbors
  4. If neighbor's indegree becomes 0, add to queue

  indegree: {0:2, 1:1, 2:0, 3:0}
  queue: [2, 3] → process 2 → queue: [3, 1]
                → process 3 → queue: [1, 0]
                → process 1 → queue: [0]
                → process 0 → done
  result: [2, 3, 1, 0]

DFS-BASED:
  1. DFS from each unvisited node
  2. After processing all neighbors, append to result
  3. Reverse at the end
```

In [ ]:
def topological_sort_kahn(numCourses, prerequisites):
    graph = defaultdict(list)
    indegree = [0] * numCourses

    for course, prereq in prerequisites:
        graph[prereq].append(course)
        indegree[course] += 1

    print(f"  indegree: {indegree}")

    queue = deque(i for i in range(numCourses) if indegree[i] == 0)
    print(f"  start queue (indegree=0): {list(queue)}")
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in graph[node]:
            indegree[neighbor] -= 1
            if indegree[neighbor] == 0:
                queue.append(neighbor)
        print(f"  process {node} → queue: {list(queue)}  order: {order}")

    if len(order) != numCourses:
        return []
    return order

print("=== Kahn's Topological Sort ===")
prereqs = [[1,2],[0,1],[0,3]]
result = topological_sort_kahn(4, prereqs)
print(f"  Topological order: {result}")

In [ ]:
def topological_sort_dfs(numCourses, prerequisites):
    graph = defaultdict(list)
    for course, prereq in prerequisites:
        graph[prereq].append(course)

    visited = set()
    stack = []

    def dfs(node, depth=0):
        visited.add(node)
        indent = "  " * (depth + 1)
        print(f"{indent}enter {node}")
        for neighbor in graph[node]:
            if neighbor not in visited:
                dfs(neighbor, depth + 1)
        stack.append(node)
        print(f"{indent}finish {node} → push to stack")

    for i in range(numCourses):
        if i not in visited:
            dfs(i)

    result = stack[::-1]
    return result

print("=== DFS Topological Sort ===")
result = topological_sort_dfs(4, [[1,2],[0,1],[0,3]])
print(f"  Topological order: {result}")

---

## Problem 5: Dijkstra's Algorithm

Shortest path in a **weighted** graph (no negative weights).  
Uses a **min-heap** (priority queue). Greedy: always process the **closest unvisited** node.

```
     A --1-- B
     |       |
     4       2
     |       |
     C --3-- D

Dijkstra from A:

  Step 0: dist = {A:0, B:∞, C:∞, D:∞}
          heap: [(0, A)]

  Step 1: pop A (cost=0)
          update B: 0+1=1 < ∞   → dist[B]=1
          update C: 0+4=4 < ∞   → dist[C]=4
          heap: [(1,B), (4,C)]

  Step 2: pop B (cost=1)
          update D: 1+2=3 < ∞   → dist[D]=3
          heap: [(3,D), (4,C)]

  Step 3: pop D (cost=3)
          check C: 3+3=6 > 4    → no update
          heap: [(4,C)]

  Step 4: pop C (cost=4)
          no unvisited neighbors

  Final: {A:0, B:1, C:4, D:3}
  Shortest paths:
    A→B: 1 (direct)
    A→D: 3 (A→B→D)
    A→C: 4 (direct, NOT A→B→D→C=6)
```

In [ ]:
import heapq

def dijkstra(graph, start):
    dist = {node: float('inf') for node in graph}
    dist[start] = 0
    prev = {node: None for node in graph}
    heap = [(0, start)]
    visited = set()

    while heap:
        cost, node = heapq.heappop(heap)
        if node in visited:
            continue
        visited.add(node)
        print(f"  pop {node} (cost={cost})")

        for neighbor, weight in graph[node]:
            new_dist = cost + weight
            if new_dist < dist[neighbor]:
                old = dist[neighbor]
                dist[neighbor] = new_dist
                prev[neighbor] = node
                heapq.heappush(heap, (new_dist, neighbor))
                sym = '∞' if old == float('inf') else old
                print(f"    update {neighbor}: {new_dist} < {sym}")

    return dist, prev

def reconstruct_path(prev, start, end):
    path = []
    node = end
    while node is not None:
        path.append(node)
        node = prev[node]
    return path[::-1]

weighted_graph = {
    "A": [("B", 1), ("C", 4)],
    "B": [("A", 1), ("D", 2)],
    "C": [("A", 4), ("D", 3)],
    "D": [("B", 2), ("C", 3)],
}

print("=== Dijkstra from A ===")
dist, prev = dijkstra(weighted_graph, "A")

print("\n  Shortest distances:")
for node in sorted(dist):
    path = reconstruct_path(prev, "A", node)
    print(f"    A → {node}: cost={dist[node]}  path={'→'.join(path)}")

---

## Problem 6: Union-Find (Disjoint Set Union)

Tracks which elements belong to the **same group**.  
Two operations: **find** (which group?) and **union** (merge groups).  
Optimizations: **path compression** + **union by rank**.

```
Initially: {0}, {1}, {2}, {3}, {4}   (each in own group)

union(0, 1):   {0,1}, {2}, {3}, {4}
union(2, 3):   {0,1}, {2,3}, {4}
union(0, 3):   {0,1,2,3}, {4}

find(2) == find(1)?  Yes (same group)
find(0) == find(4)?  No  (different groups)

Tree representation:

  After union(0,1):    0          After union(0,3):    0
                       |                              /|\\
                       1                             1 2 3

  Path compression: on find(3), point 3 directly to root
```

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
        self.components = n

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        self.components -= 1
        return True

    def connected(self, x, y):
        return self.find(x) == self.find(y)

uf = UnionFind(5)
print("=== Union-Find Trace ===")
print(f"  Initial parents: {uf.parent}  components: {uf.components}")

ops = [(0,1), (2,3), (0,3)]
for a, b in ops:
    uf.union(a, b)
    groups = defaultdict(list)
    for i in range(5):
        groups[uf.find(i)].append(i)
    print(f"  union({a},{b}) → parents: {uf.parent}  groups: {dict(groups)}")

print()
print(f"  connected(2, 1)? {uf.connected(2, 1)}")
print(f"  connected(0, 4)? {uf.connected(0, 4)}")
print(f"  components: {uf.components}")

---

## Problem 7: Number of Connected Components — LC #323

Given `n` nodes labeled `0` to `n-1` and a list of undirected edges,  
find the number of connected components.

```
n=5, edges=[[0,1],[1,2],[3,4]]

  0 - 1 - 2       3 - 4

  Component 1: {0,1,2}
  Component 2: {3,4}
  Answer: 2
```

In [ ]:
def countComponents_uf(n, edges):
    uf = UnionFind(n)
    for u, v in edges:
        uf.union(u, v)
    return uf.components

def countComponents_dfs(n, edges):
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)

    visited = set()
    count = 0

    def dfs(node):
        visited.add(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                dfs(neighbor)

    for i in range(n):
        if i not in visited:
            count += 1
            print(f"  Component {count}: starting DFS from node {i}")
            dfs(i)

    return count

edges_list = [[0,1],[1,2],[3,4]]

print("=== Connected Components (Union-Find) ===")
print(f"  Components: {countComponents_uf(5, edges_list)}")

print()
print("=== Connected Components (DFS) ===")
print(f"  Components: {countComponents_dfs(5, edges_list)}")

---

## Problem 8: Word Ladder — LC #127

Given `beginWord`, `endWord`, and a `wordList`, find the length of the **shortest transformation sequence** from `beginWord` to `endWord`, where each step changes exactly one letter and every intermediate word must be in `wordList`.

```
beginWord = "hit"
endWord   = "cog"
wordList  = ["hot","dot","dog","lot","log","cog"]

Word graph (edges = differ by 1 letter):

  hit → hot → dot → dog → cog
              ↓         ↗
             lot → log

BFS from "hit":
  Level 1: [hit]
  Level 2: [hot]
  Level 3: [dot, lot]
  Level 4: [dog, log]
  Level 5: [cog]        ← found!

  Answer: 5
```

BFS guarantees shortest path. The trick is efficiently finding neighbors  
(words that differ by one letter) using wildcard patterns: `h*t → hit, hot, hat, ...`

In [ ]:
def ladderLength(beginWord, endWord, wordList):
    word_set = set(wordList)
    if endWord not in word_set:
        return 0

    patterns = defaultdict(list)
    for word in word_set | {beginWord}:
        for i in range(len(word)):
            pattern = word[:i] + "*" + word[i+1:]
            patterns[pattern].append(word)

    visited = {beginWord}
    queue = deque([beginWord])
    level = 1

    while queue:
        level_size = len(queue)
        level_words = []

        for _ in range(level_size):
            word = queue.popleft()
            level_words.append(word)

            for i in range(len(word)):
                pattern = word[:i] + "*" + word[i+1:]
                for neighbor in patterns[pattern]:
                    if neighbor == endWord:
                        print(f"  Level {level}: {level_words}")
                        print(f"  Level {level+1}: [{endWord}] ← FOUND!")
                        return level + 1
                    if neighbor not in visited:
                        visited.add(neighbor)
                        queue.append(neighbor)

        print(f"  Level {level}: {level_words}")
        level += 1

    return 0

print("=== Word Ladder ===")
result = ladderLength("hit", "cog", ["hot","dot","dog","lot","log","cog"])
print(f"  Shortest transformation length: {result}")

---

## Practice Problems

| # | Problem | Pattern | Difficulty |
|---|---------|---------|------------|
| 200 | Number of Islands | DFS/BFS on grid | Medium |
| 133 | Clone Graph | BFS + hash map | Medium |
| 207 | Course Schedule | Cycle detection (3-color DFS) | Medium |
| 210 | Course Schedule II | Topological sort | Medium |
| 127 | Word Ladder | BFS shortest path | Hard |
| 323 | Number of Connected Components | Union-Find / DFS | Medium |
| 261 | Graph Valid Tree | Union-Find (n-1 edges, no cycle) | Medium |
| 417 | Pacific Atlantic Water Flow | Multi-source BFS/DFS | Medium |
| 743 | Network Delay Time | Dijkstra | Medium |
| 787 | Cheapest Flights Within K Stops | Modified Dijkstra / Bellman-Ford | Medium |
| 684 | Redundant Connection | Union-Find (find the cycle edge) | Medium |
| 994 | Rotting Oranges | Multi-source BFS | Medium |
| 695 | Max Area of Island | DFS on grid | Medium |
| 1091 | Shortest Path in Binary Matrix | BFS | Medium |

---

## Pattern Cheat Sheet

```
GRAPH PATTERN CHEAT SHEET:

"Shortest path (unweighted)"     → BFS
"Shortest path (weighted)"       → Dijkstra (no neg weights) / Bellman-Ford
"Connected components"           → DFS/BFS or Union-Find
"Cycle detection (directed)"     → 3-color DFS or topological sort
"Cycle detection (undirected)"   → DFS with parent tracking or Union-Find
"Ordering with dependencies"     → Topological sort (Kahn's BFS or DFS)
"Grid problems"                  → Treat as graph, DFS/BFS on cells
"Group membership"               → Union-Find
```

### Quick Decision Tree

```
Is it a graph problem?
│
├─ Shortest path?
│  ├─ Unweighted → BFS
│  └─ Weighted   → Dijkstra (or Bellman-Ford if negative weights)
│
├─ Connected components / grouping?
│  ├─ Static graph    → DFS/BFS
│  └─ Dynamic (edges added over time) → Union-Find
│
├─ Ordering / dependencies?
│  └─ Topological sort (Kahn's or DFS)
│
├─ Cycle detection?
│  ├─ Directed   → 3-color DFS
│  └─ Undirected → DFS with parent / Union-Find
│
└─ Grid traversal?
   └─ DFS/BFS treating cells as nodes, 4-directional neighbors
```

---

**Next up: Topic 11 — Dynamic Programming**